In [1]:
script = '''
import pickle
import pandas as pd
import numpy as np
import time
import os
from rdkit import Chem
from rdkit.Chem import Descriptors, AllChem
from rdkit.ML.Descriptors import MoleculeDescriptors
from chembl_webresource_client.new_client import new_client

print("Starting overnight screening...", flush=True)
start_total = time.time()

# Load models
with open("models/rf_classifier.pkl", "rb") as f:
    rf_model = pickle.load(f)
with open("models/rf_regressor.pkl", "rb") as f:
    rf_reg = pickle.load(f)
with open("models/clean_descriptors.pkl", "rb") as f:
    clean_descriptors = pickle.load(f)
with open("models/descriptor_calculator.pkl", "rb") as f:
    calculator = pickle.load(f)

descriptor_names = [d[0] for d in Descriptors.descList]
os.makedirs("screening_results", exist_ok=True)
print("Models loaded ✅", flush=True)

def extract_compounds(raw_df):
    smiles_list, chembl_ids, names = [], [], []
    for _, row in raw_df.iterrows():
        try:
            smiles = row["molecule_structures"]["canonical_smiles"]
            if smiles:
                smiles_list.append(smiles)
                chembl_ids.append(row["molecule_chembl_id"])
                names.append(row["pref_name"] if row["pref_name"]
                             else row["molecule_chembl_id"])
        except:
            pass
    return pd.DataFrame({
        "chembl_id": chembl_ids,
        "drug_name": names,
        "smiles": smiles_list
    }).drop_duplicates(subset=["chembl_id"]).reset_index(drop=True)


def screen_and_save(compounds_df, library_name, top_n=20):
    print(f"\\nScreening {library_name} ({len(compounds_df)} compounds)...",
          flush=True)
    t0 = time.time()

    desc_list, fp_list, valid_idx = [], [], []

    for i, smiles in enumerate(compounds_df["smiles"]):
        mol = Chem.MolFromSmiles(smiles)
        if mol is not None:
            descs = calculator.CalcDescriptors(mol)
            fp = AllChem.GetMorganFingerprintAsBitVect(
                mol, radius=2, nBits=2048)
            desc_list.append(descs)
            fp_list.append(list(fp))
            valid_idx.append(i)
        if (i+1) % 2000 == 0:
            elapsed = (time.time()-t0)/60
            print(f"  {i+1}/{len(compounds_df)} processed "
                  f"({elapsed:.1f} mins elapsed)", flush=True)

    desc_df = pd.DataFrame(desc_list, columns=descriptor_names)
    fp_df   = pd.DataFrame(fp_list,
                           columns=[f"Morgan_{i}" for i in range(2048)])

    X_desc = desc_df[clean_descriptors].copy()
    X_desc = X_desc.replace([np.inf, -np.inf], np.nan)
    X_desc = X_desc.fillna(X_desc.median())
    X_desc = X_desc.clip(-np.finfo(np.float32).max,
                          np.finfo(np.float32).max)
    X_desc = X_desc.astype(np.float32)

    X_comb = pd.concat([
        X_desc.reset_index(drop=True),
        fp_df.reset_index(drop=True)
    ], axis=1)
    X_comb = X_comb.replace([np.inf, -np.inf], np.nan)
    X_comb = X_comb.fillna(X_comb.median())
    X_comb = X_comb.clip(-np.finfo(np.float32).max,
                          np.finfo(np.float32).max)
    X_comb = X_comb.astype(np.float32)

    activity_pred = rf_model.predict(X_desc)
    activity_prob = rf_model.predict_proba(X_desc)[:, 1]
    pic50_pred    = rf_reg.predict(X_comb)

    results = compounds_df.iloc[valid_idx].reset_index(drop=True).copy()
    results["predicted_activity"] = activity_pred
    results["active_probability"] = activity_prob.round(4)
    results["predicted_pIC50"]    = pic50_pred.round(3)

    actives = results[results["predicted_activity"] == 1].copy()
    actives = actives.sort_values(
        "active_probability", ascending=False
    ).reset_index(drop=True)

    safe = library_name.lower().replace(" ", "_")
    actives.to_csv(
        f"screening_results/{safe}_all_hits.csv", index=False)
    actives.head(top_n).to_csv(
        f"screening_results/{safe}_top{top_n}.csv", index=False)

    elapsed = (time.time()-t0)/60
    print(f"Done! Valid: {len(valid_idx)} | "
          f"Actives: {len(actives)} | "
          f"Hit rate: {len(actives)/len(valid_idx)*100:.1f}% | "
          f"Time: {elapsed:.1f} mins", flush=True)
    return actives


# LIBRARY 1 — NATURAL COMPOUNDS
print("\\n[1/2] Fetching natural compounds...", flush=True)
natural_raw = pd.DataFrame.from_records(
    new_client.molecule.filter(
        natural_product=1,
        molecule_type="Small molecule"
    ).only(["molecule_chembl_id", "pref_name", "molecule_structures"])
)
natural_compounds = extract_compounds(natural_raw)
print(f"Fetched: {len(natural_compounds)}", flush=True)
screen_and_save(natural_compounds, "Natural Compounds")


# LIBRARY 2 — ANTIBACTERIALS
print("\\n[2/2] Fetching antibacterial compounds...", flush=True)
antibacterial_raw = pd.DataFrame.from_records(
    new_client.molecule.filter(
        molecule_type="Small molecule",
        atc_classifications__level1="J"
    ).only(["molecule_chembl_id", "pref_name", "molecule_structures"])
)
antibacterial_compounds = extract_compounds(antibacterial_raw)
print(f"Fetched: {len(antibacterial_compounds)}", flush=True)
screen_and_save(antibacterial_compounds, "Antibacterial Compounds")


# DONE
total = (time.time()-start_total)/60
print(f"\\nAll screening complete in {total:.1f} mins ✅")
print("Results saved in screening_results/")
'''

# Save script to file
with open('overnight_screening.py', 'w', encoding='utf-8') as f:
    f.write(script)

print("Script saved as overnight_screening.py ✅")

Script saved as overnight_screening.py ✅


In [2]:
exec(open('overnight_screening.py', encoding='utf-8').read())
```

**Step 3 — Leave that new notebook tab running**

---

## Result
```
New notebook tab    → running overnight script 🌙
Current notebook    → free for you to continue ✅

SyntaxError: invalid character '—' (U+2014) (3799287546.py, line 4)

In [ ]:
exec(open('overnight_screening.py', encoding='utf-8').read())

C:\Users\DELL\anaconda3\envs\tox21\lib\site-packages\chembl_webresource_client\__init__.py:4: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  __version__ = __import__('pkg_resources').get_distribution('chembl_webresource_client').version


Starting overnight screening...
Models loaded ✅

[1/2] Fetching natural compounds...
Fetched: 83119

Screening Natural Compounds (83119 compounds)...


[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerator
[13:23:26] DEPRECATION WARNING: please use MorganGenerat

  2000/83119 processed (0.3 mins elapsed)


[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerator
[13:23:43] DEPRECATION WARNING: please use MorganGenerat

...

[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:25] DEPRECATION WARNING: please use MorganGenerator
[13:30:26] DEPRECATION WARNING: please use MorganGenerator
[13:30:26] DEPRECATION WARNING: please use MorganGenerat

  36000/83119 processed (7.4 mins elapsed)


[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerator
[13:30:50] DEPRECATION WARNING: please use MorganGenerat

  38000/83119 processed (7.8 mins elapsed)


[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerator
[13:31:16] DEPRECATION WARNING: please use MorganGenerat

In [4]:
import pandas as pd

# Load natural compounds top 20
natural_top = pd.read_csv('screening_results/natural_compounds_top20.csv')

# Load antibacterial top 20
antibacterial_top = pd.read_csv(
    'screening_results/antibacterial_compounds_top20.csv'
)

print("="*55)
print("TOP 20 — NATURAL COMPOUND LpxC HITS")
print("="*55)
print(natural_top[['drug_name', 'active_probability', 
                    'predicted_pIC50']].to_string(index=True))

print("\n")
print("="*55)
print("TOP 20 — ANTIBACTERIAL LpxC HITS")
print("="*55)
print(antibacterial_top[['drug_name', 'active_probability',
                          'predicted_pIC50']].to_string(index=True))

TOP 20 — NATURAL COMPOUND LpxC HITS
        drug_name  active_probability  predicted_pIC50
0   CHEMBL2377693              0.9600            7.082
1    CHEMBL260091              0.9600            7.082
2    DALFOPRISTIN              0.9400            8.405
3   CHEMBL3577049              0.9300            8.341
4   CHEMBL1576976              0.9300            8.137
5      NIKKOMYCIN              0.9300            8.137
6   CHEMBL1791412              0.9300            7.961
7   CHEMBL1977050              0.9300            8.098
8   CHEMBL3707046              0.9300            7.993
9     CEFMEPIDIUM              0.9300            8.177
10   CHEMBL222649              0.9243            8.039
11   CHEMBL486232              0.9243            8.039
12   CHEMBL203224              0.9211            8.390
13      RELACATIB              0.9211            8.390
14   CHEMBL202991              0.9211            8.390
15   CHEMBL377713              0.9211            8.390
16  CHEMBL4211562            